# Skim Log Summary

Summarize `.out` files inside log folders whose names contain `_skim_`.

Columns:

- `folder_path`: full path of the skim log folder.
- `total_file_count`: number of `.out` files found directly inside that folder.
- `success_count`: number of `.out` files containing `SUCCESS`.
- `skipped_count`: number of `.out` files containing `SKIPPED`.
- `success_plus_skipped_count`: `success_count + skipped_count`.
- `remaining`: `total_file_count - success_plus_skipped_count`; files that are neither successful nor skipped.


In [1]:
from pathlib import Path
import os
import pandas as pd

In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)


In [ ]:
def summarize_skim_logs(root="/home/kbas/scratch"):
    """Return one row per *_skim_* log folder under root.

    The directory walk only recurses through folders. Files are inspected only
    inside folders whose name contains '_skim_', and only when they end in .out.
    """
    root = Path(root)
    rows = []

    def iter_dirs(base):
        stack = [Path(base)]
        while stack:
            current = stack.pop()
            try:
                with os.scandir(current) as entries:
                    for entry in entries:
                        if entry.is_dir(follow_symlinks=False):
                            path = Path(entry.path)
                            stack.append(path)
                            yield path
            except PermissionError:
                continue
            except FileNotFoundError:
                continue

    for folder in iter_dirs(root):
        if "_skim_" not in folder.name:
            continue

        total = 0
        success = 0
        skipped = 0

        try:
            with os.scandir(folder) as entries:
                for entry in entries:
                    if not entry.is_file(follow_symlinks=False):
                        continue
                    if not entry.name.endswith(".out"):
                        continue

                    total += 1
                    try:
                        text = Path(entry.path).read_text(errors="ignore")
                    except OSError:
                        text = ""

                    if "SUCCESS" in text:
                        success += 1
                    if "SKIPPED" in text:
                        skipped += 1
        except PermissionError:
            continue
        except FileNotFoundError:
            continue

        success_plus_skipped = success + skipped

        rows.append({
            "folder_path": str(folder),
            "total_file_count": total,
            "success_count": success,
            "skipped_count": skipped,
            "success_plus_skipped_count": success_plus_skipped,
            "remaining": total - success_plus_skipped,
        })

    return pd.DataFrame(rows).sort_values("folder_path").reset_index(drop=True)


In [4]:


skim_log_summary = summarize_skim_logs()


In [ ]:
skim_log_summary_no_spring_no_340 = skim_log_summary[
    ~skim_log_summary["folder_path"].str.contains("Spring2026MC|340StringMC", na=False)
].reset_index(drop=True)

skim_log_summary_no_spring_no_340




,folder_path,total_file_count,success_count,skipped_count,success_plus_skipped_count,remaining
0,/home/kbas/scratch/String340MC/Logs/Electron_skim_102_string,6875,6867,8,6875,0
1,/home/kbas/scratch/String340MC/Logs/Muon_skim_102_string,9811,9806,5,9811,0
2,/home/kbas/scratch/String340MC/Logs/NC_skim_102_string,9979,9977,2,9979,0
3,/home/kbas/scratch/String340MC/Logs/Tau_skim_102_string,9985,9984,1,9985,0
